# Ingeniería de Características (Feature Engineering)
## Proyecto de Tesis: Sistema inteligente para la detección y predicción de degradación de conectividad WiFi
**Autor:** Juan Vásquez  
**Universidad:** UDLA — Maestría en Inteligencia Artificial Aplicada  
**Fecha:** Marzo 2026

### Descripción
Este notebook transforma el dataset limpio mediante la creación de ventanas 
temporales y el etiquetado hacia adelante (forward labeling) para preparar 
los datos para el entrenamiento del modelo predictivo.

In [1]:
#IMPORTACION DE LIBRERIAS Y CARGA DEL DATASET LIMPIADO ANTERIORMENTE
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\juanj\Desktop\Maestría\Tesis\Datasets\dataset_qos_limpio.csv")
print(df.shape)
print(df.head())

(4584, 11)
             timestamp  latencia_rtt_ms  jitter_ms  perdida_paquetes_pct  \
0  2026-03-15 15:14:00           57.864     93.708                  10.0   
1  2026-03-15 15:14:30           51.559     94.494                  10.0   
2  2026-03-15 15:15:00           51.714     94.299                  10.0   
3  2026-03-15 15:15:30           17.813      0.187                   0.0   
4  2026-03-15 15:16:00           18.133      0.652                   0.0   

   throughput_descarga_mbps  estado_latencia  estado_jitter  estado_perdida  \
0                    36.067                0              1               1   
1                    35.648                0              1               1   
2                    42.567                0              1               1   
3                    33.446                0              0               0   
4                    43.511                0              0               0   

   estado_throughput  num_umbrales_superados etiqueta_con

## 1. Parámetros de la ventana temporal

Se definen los parámetros de la ventana temporal consistentes con el horizonte 
de predicción de 15 minutos establecido en los objetivos del proyecto:

- **Ventana de entrada**: 10 muestras × 30 segundos = 5 minutos de historial
- **Horizonte de predicción**: 30 muestras × 30 segundos = 15 minutos hacia adelante

In [2]:
VENTANA = 10        # muestras hacia atrás (5 minutos de historial)
HORIZONTE = 30      # muestras hacia adelante (15 minutos de predicción)

metricas = ["latencia_rtt_ms", "jitter_ms", "perdida_paquetes_pct", "throughput_descarga_mbps"]

## 2. Creación de ventanas temporales

Se crean 40 nuevas columnas (10 muestras × 4 métricas) con los valores 
históricos de cada métrica para que el modelo pueda aprender tendencias 
temporales en lugar de solo el valor instantáneo.

In [3]:
for i in range(1, VENTANA + 1):
    for metrica in metricas:
        df[f"{metrica}_t-{i}"] = df[metrica].shift(i)

print(df.shape)

(4584, 51)


## 3. Etiquetado hacia adelante (Forward Labeling)

Se reetiqueta cada fila con el estado de conectividad que ocurrirá 30 muestras 
(15 minutos) en el futuro. Esto convierte el problema de clasificación del estado 
actual en un problema de predicción del estado futuro.

In [4]:
df["etiqueta_futura"] = df["etiqueta_conectividad"].shift(-HORIZONTE)
df[["timestamp", "etiqueta_conectividad", "etiqueta_futura"]].head(40)

,timestamp,etiqueta_conectividad,etiqueta_futura
0,2026-03-15 15:14:00,degradada,normal
1,2026-03-15 15:14:30,degradada,normal
2,2026-03-15 15:15:00,degradada,normal
3,2026-03-15 15:15:30,normal,normal
4,2026-03-15 15:16:00,normal,normal
5,2026-03-15 15:16:30,normal,normal
6,2026-03-15 15:17:00,normal,normal
7,2026-03-15 15:17:30,normal,normal
8,2026-03-15 15:18:00,degradada,normal
9,2026-03-15 15:18:30,normal,normal


## 4. Eliminación de filas incompletas

Se eliminan las filas sin historial completo (primeras 10) y sin etiqueta 
futura (últimas 30) resultantes de las transformaciones anteriores.

In [5]:
df_features = df.dropna().reset_index(drop=True)
print(df_features.shape)
print(df_features["etiqueta_futura"].value_counts())

(4544, 52)
etiqueta_futura
normal       3129
degradada     893
critica       522
Name: count, dtype: int64


## 5. Estadísticos de ventana deslizante

Se calculan media, desviación estándar, máximo y mínimo de cada métrica
en ventanas de 5 minutos (10 muestras) y 15 minutos (30 muestras) para
capturar el comportamiento estadístico previo a cada muestra, complementando
los valores puntuales del shift.

In [6]:
ventanas = [10, 30]  # 10 muestras = 5 min, 30 muestras = 15 min

for v in ventanas:
    for metrica in metricas:
        df_features[f"{metrica}_mean_{v}"] = df_features[metrica].rolling(v).mean()
        df_features[f"{metrica}_std_{v}"]  = df_features[metrica].rolling(v).std()
        df_features[f"{metrica}_max_{v}"]  = df_features[metrica].rolling(v).max()
        df_features[f"{metrica}_min_{v}"]  = df_features[metrica].rolling(v).min()

print(df_features.shape)

(4544, 84)


## 6. Exportación del dataset final

Se exporta el dataset con ventanas temporales y forward labeling listo para el entrenamiento del modelo.

In [7]:
df_features.to_csv(r"C:\Users\juanj\Desktop\Maestría\Tesis\Datasets\Dataset_final.csv", index = False)